In [7]:
!pip install -q streamlit pydicom opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 108.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 100.1 MB/s eta 0:00:0000:010:01


In [8]:
import pathlib

import numpy as np
import pandas as pd
import cv2
import pydicom

import torch
import torch.nn as nn
import torchvision.models as models

import streamlit as st

In [9]:
TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

NUM_CLASSES = len(TARGETS)

In [10]:
THRESHOLDS = {
    "ACL": 0.5,
    "MCL": 0.5,
    "Medial Meniscus": 0.5,
    "Lateral Meniscus": 0.5,
    "Medial OA": 0.5,
    "Lateral OA": 0.5,
    "PF OA": 0.5,
    "Effusion": 0.5,
    "Synovitis": 0.5,
    "Baker's": 0.5,
    "Contusion": 0.5,
    "Fracture": 0.5
}

In [11]:
def load_dicom(path):
    """
    Read a DICOM file and return its pixel array.
    """

    ds = pydicom.dcmread(path)

    pixel_array = ds.pixel_array.astype(np.float32)

    return ds, pixel_array

In [12]:
def sort_slices(dicom_files):
    """
    Sort DICOM slices in anatomical order.
    """

    datasets = []

    for file_path in dicom_files:
        ds = pydicom.dcmread(file_path)
        datasets.append(ds)

    datasets.sort(
        key=lambda ds: int(
            getattr(ds, "InstanceNumber", 0)
        )
    )

    return datasets

In [13]:
def select_uniform_slices(datasets, num_slices=16):
    """
    Select uniformly distributed slices from a series.
    """

    total_slices = len(datasets)

    if total_slices <= num_slices:
        return datasets

    indices = np.linspace(
        0,
        total_slices - 1,
        num_slices,
        dtype=int
    )

    return [datasets[i] for i in indices]

In [14]:
def preprocess_slice(ds):
    """
    Apply the exact slice preprocessing pipeline.
    """

    image = ds.pixel_array.astype(np.float32)

    # 1st / 99th percentile
    p1 = np.percentile(image, 1)
    p99 = np.percentile(image, 99)

    # Clip
    image = np.clip(image, p1, p99)

    # Normalize to 0-1
    if p99 > p1:
        image = (image - p1) / (p99 - p1)
    else:
        image = np.zeros_like(image)

    # Resize
    image = cv2.resize(
        image,
        (224, 224),
        interpolation=cv2.INTER_LINEAR
    )

    # Grayscale → 3 channels
    image = np.stack(
        [image, image, image],
        axis=0
    )

    # float32
    image = image.astype(np.float32)

    return image

In [15]:
def preprocess_series(dicom_files, num_slices=16):
    """
    Prepare one MRI series for the model.
    """

    datasets = sort_slices(dicom_files)

    selected = select_uniform_slices(
        datasets,
        num_slices
    )

    slices = []

    for ds in selected:
        image = preprocess_slice(ds)
        slices.append(image)

    return np.stack(slices)

In [16]:
class AttentionPool(nn.Module):

    def __init__(self, feat_dim, hidden_dim=128):
        super().__init__()

        self.attn = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, mask=None):

        scores = self.attn(x).squeeze(-1)

        if mask is not None:
            scores = scores.masked_fill(
                mask == 0,
                float("-inf")
            )

        weights = torch.softmax(
            scores,
            dim=1
        )

        return torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )

In [17]:
def build_resnet50_backbone(
    pretrained=False,
    radimagenet_path=None
):

    resnet = models.resnet50(weights=None)

    if radimagenet_path:

        state_dict = torch.load(
            radimagenet_path,
            map_location="cpu"
        )

        resnet.load_state_dict(
            state_dict,
            strict=False
        )

    elif pretrained:

        resnet = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT
        )

    return nn.Sequential(
        *list(resnet.children())[:-1]
    )

In [18]:
class KneeAbnormalityModelMIL(nn.Module):

    def __init__(
        self,
        num_classes=12,
        pretrained=False,
        radimagenet_path=None
    ):

        super().__init__()

        self.backbone = build_resnet50_backbone(
            pretrained=pretrained,
            radimagenet_path=radimagenet_path
        )

        self.slice_pool = AttentionPool(2048)

        self.series_pool = AttentionPool(2048)

        self.dropout = nn.Dropout(p=0.4)

        self.fc = nn.Linear(
            2048,
            num_classes
        )

    def forward(self, x, series_mask):

        b, s, k, c, h, w = x.shape

        x = x.view(
            b * s * k,
            c,
            h,
            w
        )

        features = self.backbone(x).view(
            b,
            s,
            k,
            2048
        )

        series_embeds = self.slice_pool(
            features.view(
                b * s,
                k,
                2048
            )
        ).view(
            b,
            s,
            2048
        )

        study_embed = self.series_pool(
            series_embeds,
            mask=series_mask
        )

        study_embed = self.dropout(
            study_embed
        )

        return self.fc(study_embed)

In [19]:
MODEL_DIR = pathlib.Path(
    "/kaggle/input/models/amraldswqy135/knee-model2/pytorch/default/1"
)

CHECKPOINTS = [
    MODEL_DIR / "/kaggle/input/models/amraldswqy135/knee-model2/pytorch/default/1/knee_model_mil_v3_fold0_best.pt",
    MODEL_DIR / "/kaggle/input/models/amraldswqy135/knee-model2/pytorch/default/1/knee_model_mil_v3_fold1_best.pt"
]

In [20]:
@st.cache_resource
def load_models():

    loaded_models = []

    for checkpoint_path in CHECKPOINTS:

        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False
        )

        model = KneeAbnormalityModelMIL(
            num_classes=NUM_CLASSES,
            pretrained=False,
            radimagenet_path=None
        ).to(DEVICE)

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        model.eval()

        loaded_models.append(model)

    return loaded_models

In [21]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [22]:
models_list = load_models()

2026-08-13 17:25:39.717 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [23]:
@st.cache_data
def load_series_metadata(path):
    return pd.read_csv(path)

2026-08-13 17:25:43.563 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [24]:
def build_series_lookup(series_meta_df):

    lookup = {}

    for _, row in series_meta_df.iterrows():

        uid = str(row["SeriesInstanceUID"])

        lookup[uid] = {
            "Fluid_Sensitive": row.get("Fluid_Sensitive", 0) or 0,
            "Fat_Suppression": row.get("Fat_Suppression", 0) or 0,
            "Anatomical_Plane": row.get("Anatomical_Plane", "")
        }

    return lookup

In [25]:
PLANE_PRIORITY = {
    "Sagittal": 3,
    "Coronal": 2,
    "Axial": 1
}

In [26]:
def score_series(series_uid, series_lookup):

    meta = series_lookup.get(series_uid)

    if meta is None:
        return -1

    return (
        meta["Fluid_Sensitive"] * 10
        + meta["Fat_Suppression"] * 5
        + PLANE_PRIORITY.get(
            meta["Anatomical_Plane"],
            0
        )
    )

In [27]:
def select_best_series(series_folders, series_lookup):

    best_by_plane = {}

    for series_path in series_folders:

        series_uid = series_path.name

        meta = series_lookup.get(series_uid)

        plane = (
            meta["Anatomical_Plane"]
            if meta
            else "Unknown"
        )

        score = score_series(
            series_uid,
            series_lookup
        )

        if (
            plane not in best_by_plane
            or score > best_by_plane[plane][0]
        ):
            best_by_plane[plane] = (
                score,
                series_path
            )

    selected = []

    for plane in [
        "Sagittal",
        "Coronal",
        "Axial"
    ]:

        if plane in best_by_plane:

            selected.append(
                best_by_plane[plane][1]
            )

    return selected[:3]

In [28]:
def get_sorted_dicom_files(series_path):

    dicom_files = list(
        series_path.glob("*.dcm")
    )

    headers = []

    for file_path in dicom_files:

        try:

            ds = pydicom.dcmread(
                file_path,
                stop_before_pixels=True
            )

            if hasattr(ds, "InstanceNumber"):

                key = int(
                    getattr(
                        ds,
                        "InstanceNumber",
                        0
                    )
                )

            else:

                key = float(
                    getattr(
                        ds,
                        "ImagePositionPatient",
                        [0, 0, 0]
                    )[2]
                )

            headers.append(
                (key, file_path)
            )

        except Exception:
            continue

    headers.sort(
        key=lambda x: x[0]
    )

    return [
        file_path
        for _, file_path in headers
    ]

In [29]:
def select_16_slices(dicom_files):

    n = len(dicom_files)

    if n == 0:
        return []

    indices = np.linspace(
        0,
        n - 1,
        num=16,
        dtype=int
    )

    return [
        dicom_files[i]
        for i in indices
    ]

In [30]:
def load_selected_images(selected_files):

    images = []

    for file_path in selected_files:

        try:

            ds = pydicom.dcmread(file_path)

            image = ds.pixel_array.astype(
                np.float32
            )

            images.append(image)

        except Exception:

            images.append(None)

    valid = [
        image
        for image in images
        if image is not None
    ]

    if not valid:
        return None

    fill_shape = valid[0].shape

    images = [
        image
        if image is not None
        else np.zeros(
            fill_shape,
            dtype=np.float32
        )
        for image in images
    ]

    return np.array(images)

In [31]:
def preprocess_volume(volume):

    # float32
    volume = volume.astype(
        np.float32
    )

    # 1st / 99th percentile
    p1, p99 = np.percentile(
        volume,
        (1, 99)
    )

    # Same denominator logic as training
    denom = (
        (p99 - p1)
        if (p99 - p1) > 1e-5
        else 1.0
    )

    # Clip
    volume = np.clip(
        volume,
        p1,
        p99
    )

    # Normalize 0 → 1
    volume = (
        volume - p1
    ) / denom

    # Resize every slice
    volume = np.array([
        cv2.resize(
            image,
            (224, 224),
            interpolation=cv2.INTER_AREA
        )
        for image in volume
    ], dtype=np.float32)

    return volume

In [32]:
def preprocess_study(
    study_path,
    series_lookup,
    max_series=3
):

    study_path = pathlib.Path(
        study_path
    )

    series_folders = [
        folder
        for folder in study_path.iterdir()
        if folder.is_dir()
    ]

    selected_series = select_best_series(
        series_folders,
        series_lookup
    )

    stack = np.zeros(
        (
            max_series,
            16,
            224,
            224
        ),
        dtype=np.float32
    )

    mask = np.zeros(
        (max_series,),
        dtype=np.float32
    )

    selected_info = []

    for i, series_path in enumerate(
        selected_series
    ):

        dicom_files = (
            get_sorted_dicom_files(
                series_path
            )
        )

        selected_files = select_16_slices(
            dicom_files
        )

        volume = load_selected_images(
            selected_files
        )

        if volume is None:
            continue

        volume = preprocess_volume(
            volume
        )

        stack[i] = volume

        mask[i] = 1.0

        selected_info.append({
            "SeriesInstanceUID": series_path.name
        })

    return stack, mask, selected_info

In [33]:
def prepare_model_input(stack, mask):

    stack_tensor = torch.from_numpy(
        stack
    )

    # [3, 16, 224, 224]
    #        ↓
    # [3, 16, 3, 224, 224]

    stack_tensor = (
        stack_tensor
        .unsqueeze(2)
        .repeat(1, 1, 3, 1, 1)
    )

    # Add batch dimension
    # [3, 16, 3, 224, 224]
    #        ↓
    # [1, 3, 16, 3, 224, 224]

    stack_tensor = stack_tensor.unsqueeze(0)

    mask_tensor = torch.from_numpy(
        mask
    ).unsqueeze(0)

    return (
        stack_tensor.to(DEVICE),
        mask_tensor.to(DEVICE)
    )

In [34]:
def predict_study(models_list, stack, mask):

    # Prepare input
    volumes, series_mask = prepare_model_input(
        stack,
        mask
    )

    fold_probabilities = []

    # Inference for every fold
    with torch.no_grad():

        for model in models_list:

            logits = model(
                volumes,
                series_mask
            )

            probabilities = torch.sigmoid(
                logits
            )

            fold_probabilities.append(
                probabilities.cpu().numpy()[0]
            )

    # [2, 12]
    fold_probabilities = np.array(
        fold_probabilities
    )

    # Ensemble
    ensemble_probabilities = np.mean(
        fold_probabilities,
        axis=0
    )

    return ensemble_probabilities

In [35]:
THRESHOLDS = {
    "ACL": 0.50,
    "MCL": 0.50,
    "Medial Meniscus": 0.50,
    "Lateral Meniscus": 0.50,
    "Medial OA": 0.50,
    "Lateral OA": 0.50,
    "PF OA": 0.50,
    "Effusion": 0.50,
    "Synovitis": 0.50,
    "Baker's": 0.50,
    "Contusion": 0.50,
    "Fracture": 0.50,
}

In [36]:
def apply_thresholds(probabilities):

    results = []

    for class_name, probability in zip(
        TARGETS,
        probabilities
    ):

        threshold = THRESHOLDS[class_name]

        detected = probability >= threshold

        results.append({
            "Abnormality": class_name,
            "Probability": float(probability),
            "Detected": bool(detected)
        })

    return results

In [37]:
def run_study_inference(
    study_path,
    series_lookup,
    models_list
):

    # -------------------------
    # 1. Preprocess Study
    # -------------------------

    stack, mask, selected_series = preprocess_study(
        study_path,
        series_lookup,
        max_series=3
    )

    # -------------------------
    # 2. Model Prediction
    # -------------------------

    probabilities = predict_study(
        models_list,
        stack,
        mask
    )

    # -------------------------
    # 3. Apply Rules
    # -------------------------

    results = apply_thresholds(
        probabilities
    )

    return {
        "probabilities": probabilities,
        "results": results,
        "selected_series": selected_series,
        "series_mask": mask
    }

**Stystem UI**

In [38]:
# st.set_page_config(
#     page_title="Knee MRI Abnormality Detection",
#     page_icon="",
#     layout="wide"
# )

In [39]:
# st.title(
#     " Knee MRI Abnormality Detection"
# )

# st.write(
#     "Upload a Knee MRI Study to detect "
#     "multiple abnormalities."
# )

In [40]:
# uploaded_file = st.file_uploader(
#     "Upload Knee MRI Study",
#     type=["zip"]
# )

In [41]:
import zipfile
import tempfile
def extract_study(uploaded_file):

    temp_dir = tempfile.mkdtemp()

    zip_path = pathlib.Path(temp_dir) / "study.zip"

    with open(zip_path, "wb") as f:
        f.write(uploaded_file.getbuffer())

    extract_dir = pathlib.Path(temp_dir) / "study"

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as zip_ref:

        zip_ref.extractall(extract_dir)

    return extract_dir

In [42]:
# if uploaded_file is not None:

#     if st.button(
#         "🔍 Analyze Study",
#         type="primary"
#     ):

#         with st.spinner(
#             "Analyzing Knee MRI..."
#         ):

#             study_root = extract_study(
#                 uploaded_file
#             )

#             output = run_study_inference(
#                 study_root,
#                 series_lookup,
#                 models_list
#             )

In [43]:
COMP_ROOT = pathlib.Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

TEST_SERIES_DIR = (
    COMP_ROOT / "test_series"
)

TEST_SERIES_META_PATH = (
    COMP_ROOT / "test_series.csv"
)

In [44]:
@st.cache_data
def load_series_metadata(path):

    return pd.read_csv(path)

2026-08-13 17:25:43.843 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [45]:
series_meta_df = load_series_metadata(
    TEST_SERIES_META_PATH
)

2026-08-13 17:25:43.857 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-08-13 17:25:43.859 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [46]:
series_lookup = build_series_lookup(
    series_meta_df
)

In [47]:
study_folders = [
    folder
    for folder in TEST_SERIES_DIR.iterdir()
    if folder.is_dir()
]

In [48]:
study_uids = sorted([
    folder.name
    for folder in study_folders
])

In [49]:
selected_study_uid = st.selectbox(
    "Select a Knee MRI Study",
    study_uids
)

2026-08-13 17:25:43.913 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:43.914 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:43.915 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:43.916 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-08-13 17:25:43.916 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:43.985 
  command:

    streamlit run /usr/local/lib/python3.12/

In [50]:
selected_study_path = (
    TEST_SERIES_DIR
    / selected_study_uid
)

In [51]:
# st.title(
#     "🦴 Knee MRI Abnormality Detection"
# )

# st.write(
#     "Select a Knee MRI Study from the RSNA test dataset."
# )

# selected_study_uid = st.selectbox(
#     "Select Study",
#     study_uids
# )

# if st.button(
#     "🔍 Analyze Study",
#     type="primary"
# ):

#     with st.spinner(
#         "Analyzing Knee MRI..."
#     ):

#         selected_study_path = (
#             TEST_SERIES_DIR
#             / selected_study_uid
#         )

#         output = run_study_inference(
#             selected_study_path,
#             series_lookup,
#             models_list
#         )

In [52]:
output = run_study_inference( selected_study_path,
    series_lookup,
    models_list)

In [53]:
st.subheader(
    "Selected MRI Series"
)

for i, series in enumerate(
    output["selected_series"],
    start=1
):

    st.write(
        f"Series {i}: "
        f"{series['SeriesInstanceUID']}"
    )

2026-08-13 17:25:48.569 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:48.571 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:48.571 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:48.572 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:48.573 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:48.574 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:48.575 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:48.576 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [54]:
results = output["results"]

In [55]:
detected = [
    result
    for result in results
    if result["Detected"]
]

In [56]:
if detected:

    st.subheader(
        "Detected Abnormalities"
    )

    for result in detected:

        st.success(
            f"{result['Abnormality']} "
            f"— {result['Probability']:.1%}"
        )

else:

    st.info(
        "No abnormalities detected "
        "above the configured thresholds."
    )

2026-08-13 17:25:48.609 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:48.610 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:48.610 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:48.829 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:48.830 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:48.830 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:48.831 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:25:48.832 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [57]:
# st.subheader(
#     "All Predictions"
# )

# for result in results:

#     st.write(
#         f"**{result['Abnormality']}**: "
#         f"{result['Probability']:.2%}"
#     )

In [58]:
# !streamlit run app.py

Test

In [59]:
# Pick the first Study
sample_study_uid = study_uids[0]

sample_study_path = TEST_SERIES_DIR / sample_study_uid

print("Testing Study:")
print(sample_study_uid)

Testing Study:
1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191


In [60]:
output = run_study_inference(
    sample_study_path,
    series_lookup,
    models_list
)

In [61]:
print("\nSelected Series:")

for series in output["selected_series"]:
    print(series["SeriesInstanceUID"])


Selected Series:
1.2.826.0.1.3680043.8.498.17811502614030631664517622518906646132
1.2.826.0.1.3680043.8.498.32856494541816845805947850304653662081
1.2.826.0.1.3680043.8.498.44334485654554877495800965096767429689


In [62]:
print("\nPredictions:")

for result in output["results"]:
    print(
        f"{result['Abnormality']:20s} "
        f"{result['Probability']:.4f} "
        f"-> "
        f"{'Detected' if result['Detected'] else 'Not Detected'}"
    )


Predictions:
ACL                  0.0142 -> Not Detected
MCL                  0.0525 -> Not Detected
Medial Meniscus      0.4543 -> Not Detected
Lateral Meniscus     0.0516 -> Not Detected
Medial OA            0.9340 -> Detected
Lateral OA           0.7032 -> Detected
PF OA                0.8916 -> Detected
Effusion             0.5971 -> Detected
Synovitis            0.3383 -> Not Detected
Baker's              0.7694 -> Detected
Contusion            0.1721 -> Not Detected
Fracture             0.0881 -> Not Detected


In [63]:
import inspect
functions_to_export = [
    load_series_metadata,
    build_series_lookup,
    score_series,
    select_best_series,
    get_sorted_dicom_files,
    select_16_slices,
    load_selected_images,
    preprocess_volume,
    preprocess_study,
    prepare_model_input,
    predict_study,
    apply_thresholds,
    run_study_inference,
    build_resnet50_backbone,
    AttentionPool,
    KneeAbnormalityModelMIL,
    load_models,
]

In [64]:
# %%writefile /kaggle/working/app.py

# import pathlib
# import numpy as np
# import pandas as pd
# import cv2
# import pydicom

# import torch
# import torch.nn as nn
# import torchvision.models as models

# import streamlit as st


# # =========================================================
# # CONFIG
# # =========================================================

# DEVICE = torch.device(
#     "cuda" if torch.cuda.is_available() else "cpu"
# )

# COMP_ROOT = pathlib.Path(
#     "/kaggle/input/competitions/rsna-knee-abnormality-detection"
# )

# TEST_SERIES_DIR = COMP_ROOT / "test_series"

# TEST_SERIES_META_PATH = COMP_ROOT / "test_series.csv"

# MODEL_DIR = pathlib.Path(
#     "/kaggle/input/models/amraldswqy135/knee-model2/pytorch/default/1"
# )


# TARGETS = [
#     "ACL",
#     "MCL",
#     "Medial Meniscus",
#     "Lateral Meniscus",
#     "Medial OA",
#     "Lateral OA",
#     "PF OA",
#     "Effusion",
#     "Synovitis",
#     "Baker's",
#     "Contusion",
#     "Fracture"
# ]

# NUM_CLASSES = len(TARGETS)

# THRESHOLDS = {
#     name: 0.50
#     for name in TARGETS
# }

# PLANE_PRIORITY = {
#     "Sagittal": 3,
#     "Coronal": 2,
#     "Axial": 1
# }


# # =========================================================
# # ATTENTION
# # =========================================================

# class AttentionPool(nn.Module):

#     def __init__(self, feat_dim, hidden_dim=128):

#         super().__init__()

#         self.attn = nn.Sequential(
#             nn.Linear(feat_dim, hidden_dim),
#             nn.Tanh(),
#             nn.Linear(hidden_dim, 1)
#         )

#     def forward(self, x, mask=None):

#         scores = self.attn(x).squeeze(-1)

#         if mask is not None:

#             scores = scores.masked_fill(
#                 mask == 0,
#                 float("-inf")
#             )

#         weights = torch.softmax(
#             scores,
#             dim=1
#         )

#         return torch.sum(
#             x * weights.unsqueeze(-1),
#             dim=1
#         )


# # =========================================================
# # RESNET50
# # =========================================================

# def build_resnet50_backbone():

#     resnet = models.resnet50(
#         weights=None
#     )

#     return nn.Sequential(
#         *list(resnet.children())[:-1]
#     )


# # =========================================================
# # MODEL
# # =========================================================

# class KneeAbnormalityModelMIL(nn.Module):

#     def __init__(self, num_classes=12):

#         super().__init__()

#         self.backbone = build_resnet50_backbone()

#         self.slice_pool = AttentionPool(2048)

#         self.series_pool = AttentionPool(2048)

#         self.dropout = nn.Dropout(p=0.4)

#         self.fc = nn.Linear(
#             2048,
#             num_classes
#         )

#     def forward(self, x, series_mask):

#         b, s, k, c, h, w = x.shape

#         x = x.view(
#             b * s * k,
#             c,
#             h,
#             w
#         )

#         features = self.backbone(x).view(
#             b,
#             s,
#             k,
#             2048
#         )

#         series_embeds = self.slice_pool(
#             features.view(
#                 b * s,
#                 k,
#                 2048
#             )
#         ).view(
#             b,
#             s,
#             2048
#         )

#         study_embed = self.series_pool(
#             series_embeds,
#             mask=series_mask
#         )

#         study_embed = self.dropout(
#             study_embed
#         )

#         return self.fc(study_embed)


# # =========================================================
# # LOAD MODELS
# # =========================================================

# @st.cache_resource
# def load_models():

#     checkpoint_paths = [
#         MODEL_DIR / "knee_model_mil_v3_fold0_best.pt",
#         MODEL_DIR / "knee_model_mil_v3_fold1_best.pt"
#     ]

#     loaded_models = []

#     for checkpoint_path in checkpoint_paths:

#         checkpoint = torch.load(
#             checkpoint_path,
#             map_location=DEVICE,
#             weights_only=False
#         )

#         model = KneeAbnormalityModelMIL(
#             num_classes=NUM_CLASSES
#         ).to(DEVICE)

#         model.load_state_dict(
#             checkpoint["model_state_dict"]
#         )

#         model.eval()

#         loaded_models.append(model)

#     return loaded_models


# # =========================================================
# # SERIES METADATA
# # =========================================================

# @st.cache_data
# def load_series_metadata(path):

#     return pd.read_csv(path)


# def build_series_lookup(series_meta_df):

#     lookup = {}

#     for _, row in series_meta_df.iterrows():

#         uid = str(
#             row["SeriesInstanceUID"]
#         )

#         lookup[uid] = {
#             "Fluid_Sensitive":
#                 row.get(
#                     "Fluid_Sensitive",
#                     0
#                 ) or 0,

#             "Fat_Suppression":
#                 row.get(
#                     "Fat_Suppression",
#                     0
#                 ) or 0,

#             "Anatomical_Plane":
#                 row.get(
#                     "Anatomical_Plane",
#                     ""
#                 )
#         }

#     return lookup


# def score_series(
#     series_uid,
#     series_lookup
# ):

#     meta = series_lookup.get(
#         series_uid
#     )

#     if meta is None:
#         return -1

#     return (
#         meta["Fluid_Sensitive"] * 10
#         +
#         meta["Fat_Suppression"] * 5
#         +
#         PLANE_PRIORITY.get(
#             meta["Anatomical_Plane"],
#             0
#         )
#     )


# def select_best_series(
#     series_folders,
#     series_lookup
# ):

#     best_by_plane = {}

#     for series_path in series_folders:

#         series_uid = series_path.name

#         meta = series_lookup.get(
#             series_uid
#         )

#         plane = (
#             meta["Anatomical_Plane"]
#             if meta
#             else "Unknown"
#         )

#         score = score_series(
#             series_uid,
#             series_lookup
#         )

#         if (
#             plane not in best_by_plane
#             or score >
#             best_by_plane[plane][0]
#         ):

#             best_by_plane[plane] = (
#                 score,
#                 series_path
#             )

#     selected = []

#     for plane in [
#         "Sagittal",
#         "Coronal",
#         "Axial"
#     ]:

#         if plane in best_by_plane:

#             selected.append(
#                 best_by_plane[plane][1]
#             )

#     return selected[:3]


# # =========================================================
# # DICOM
# # =========================================================

# def get_sorted_dicom_files(
#     series_path
# ):

#     dicom_files = list(
#         series_path.glob("*.dcm")
#     )

#     headers = []

#     for file_path in dicom_files:

#         try:

#             ds = pydicom.dcmread(
#                 file_path,
#                 stop_before_pixels=True
#             )

#             if hasattr(
#                 ds,
#                 "InstanceNumber"
#             ):

#                 key = int(
#                     getattr(
#                         ds,
#                         "InstanceNumber",
#                         0
#                     )
#                 )

#             else:

#                 key = float(
#                     getattr(
#                         ds,
#                         "ImagePositionPatient",
#                         [0, 0, 0]
#                     )[2]
#                 )

#             headers.append(
#                 (
#                     key,
#                     file_path
#                 )
#             )

#         except Exception:

#             continue

#     headers.sort(
#         key=lambda x: x[0]
#     )

#     return [
#         file_path
#         for _, file_path in headers
#     ]


# def select_16_slices(
#     dicom_files
# ):

#     n = len(dicom_files)

#     if n == 0:
#         return []

#     indices = np.linspace(
#         0,
#         n - 1,
#         num=16,
#         dtype=int
#     )

#     return [
#         dicom_files[i]
#         for i in indices
#     ]


# def load_selected_images(
#     selected_files
# ):

#     images = []

#     for file_path in selected_files:

#         try:

#             ds = pydicom.dcmread(
#                 file_path
#             )

#             image = (
#                 ds.pixel_array
#                 .astype(np.float32)
#             )

#             images.append(image)

#         except Exception:

#             images.append(None)

#     valid = [
#         image
#         for image in images
#         if image is not None
#     ]

#     if not valid:
#         return None

#     fill_shape = valid[0].shape

#     images = [
#         image
#         if image is not None
#         else np.zeros(
#             fill_shape,
#             dtype=np.float32
#         )
#         for image in images
#     ]

#     return np.array(images)


# # =========================================================
# # PREPROCESSING
# # =========================================================

# def preprocess_volume(
#     volume
# ):

#     volume = volume.astype(
#         np.float32
#     )

#     p1, p99 = np.percentile(
#         volume,
#         (1, 99)
#     )

#     denom = (
#         (p99 - p1)
#         if (p99 - p1) > 1e-5
#         else 1.0
#     )

#     volume = np.clip(
#         volume,
#         p1,
#         p99
#     )

#     volume = (
#         volume - p1
#     ) / denom

#     volume = np.array(
#         [
#             cv2.resize(
#                 image,
#                 (224, 224),
#                 interpolation=cv2.INTER_AREA
#             )
#             for image in volume
#         ],
#         dtype=np.float32
#     )

#     return volume


# def preprocess_study(
#     study_path,
#     series_lookup,
#     max_series=3
# ):

#     study_path = pathlib.Path(
#         study_path
#     )

#     series_folders = [
#         folder
#         for folder in study_path.iterdir()
#         if folder.is_dir()
#     ]

#     selected_series = select_best_series(
#         series_folders,
#         series_lookup
#     )

#     stack = np.zeros(
#         (
#             max_series,
#             16,
#             224,
#             224
#         ),
#         dtype=np.float32
#     )

#     mask = np.zeros(
#         (max_series,),
#         dtype=np.float32
#     )

#     selected_info = []

#     for i, series_path in enumerate(
#         selected_series
#     ):

#         dicom_files = (
#             get_sorted_dicom_files(
#                 series_path
#             )
#         )

#         selected_files = (
#             select_16_slices(
#                 dicom_files
#             )
#         )

#         volume = load_selected_images(
#             selected_files
#         )

#         if volume is None:
#             continue

#         volume = preprocess_volume(
#             volume
#         )

#         stack[i] = volume

#         mask[i] = 1.0

#         selected_info.append({
#             "SeriesInstanceUID":
#                 series_path.name
#         })

#     return (
#         stack,
#         mask,
#         selected_info
#     )


# # =========================================================
# # MODEL INPUT
# # =========================================================

# def prepare_model_input(
#     stack,
#     mask
# ):

#     stack_tensor = torch.from_numpy(
#         stack
#     )

#     stack_tensor = (
#         stack_tensor
#         .unsqueeze(2)
#         .repeat(
#             1,
#             1,
#             3,
#             1,
#             1
#         )
#     )

#     stack_tensor = (
#         stack_tensor
#         .unsqueeze(0)
#     )

#     mask_tensor = torch.from_numpy(
#         mask
#     ).unsqueeze(0)

#     return (
#         stack_tensor.to(DEVICE),
#         mask_tensor.to(DEVICE)
#     )


# # =========================================================
# # PREDICTION
# # =========================================================

# def predict_study(
#     models_list,
#     stack,
#     mask
# ):

#     volumes, series_mask = (
#         prepare_model_input(
#             stack,
#             mask
#         )
#     )

#     fold_probabilities = []

#     with torch.no_grad():

#         for model in models_list:

#             logits = model(
#                 volumes,
#                 series_mask
#             )

#             probabilities = (
#                 torch.sigmoid(logits)
#             )

#             fold_probabilities.append(
#                 probabilities
#                 .cpu()
#                 .numpy()[0]
#             )

#     fold_probabilities = np.array(
#         fold_probabilities
#     )

#     ensemble_probabilities = (
#         np.mean(
#             fold_probabilities,
#             axis=0
#         )
#     )

#     return ensemble_probabilities


# def apply_thresholds(
#     probabilities
# ):

#     results = []

#     for class_name, probability in zip(
#         TARGETS,
#         probabilities
#     ):

#         threshold = THRESHOLDS[
#             class_name
#         ]

#         detected = (
#             probability >= threshold
#         )

#         results.append({
#             "Abnormality":
#                 class_name,

#             "Probability":
#                 float(probability),

#             "Detected":
#                 bool(detected)
#         })

#     return results


# def run_study_inference(
#     study_path,
#     series_lookup,
#     models_list
# ):

#     stack, mask, selected_series = (
#         preprocess_study(
#             study_path,
#             series_lookup,
#             max_series=3
#         )
#     )

#     probabilities = predict_study(
#         models_list,
#         stack,
#         mask
#     )

#     results = apply_thresholds(
#         probabilities
#     )

#     return {
#         "probabilities":
#             probabilities,

#         "results":
#             results,

#         "selected_series":
#             selected_series,

#         "series_mask":
#             mask
#     }


# # =========================================================
# # STREAMLIT UI
# # =========================================================

# st.set_page_config(
#     page_title="Knee MRI Abnormality Detection",
#     page_icon="🦴",
#     layout="wide"
# )

# st.title(
#     "🦴 Knee MRI Abnormality Detection"
# )

# st.caption(
#     "Multi-Label Deep Learning Analysis "
#     "of Knee MRI Studies"
# )


# # Load metadata
# series_meta_df = load_series_metadata(
#     TEST_SERIES_META_PATH
# )

# series_lookup = build_series_lookup(
#     series_meta_df
# )


# # Load studies
# study_folders = [
#     folder
#     for folder in TEST_SERIES_DIR.iterdir()
#     if folder.is_dir()
# ]

# study_uids = sorted([
#     folder.name
#     for folder in study_folders
# ])


# # Load models
# models_list = load_models()


# # Study selection
# selected_study_uid = st.selectbox(
#     "Select a Knee MRI Study",
#     study_uids
# )


# if st.button(
#     "🔍 Analyze Study",
#     type="primary"
# ):

#     selected_study_path = (
#         TEST_SERIES_DIR
#         / selected_study_uid
#     )

#     with st.spinner(
#         "Analyzing Knee MRI..."
#     ):

#         output = run_study_inference(
#             selected_study_path,
#             series_lookup,
#             models_list
#         )


#     # -----------------------------------------
#     # Selected Series
#     # -----------------------------------------

#     st.subheader(
#         "Selected MRI Series"
#     )

#     for i, series in enumerate(
#         output["selected_series"],
#         start=1
#     ):

#         st.write(
#             f"**Series {i}:** "
#             f"{series['SeriesInstanceUID']}"
#         )


#     # -----------------------------------------
#     # Detected Abnormalities
#     # -----------------------------------------

#     results = output["results"]

#     detected = [
#         result
#         for result in results
#         if result["Detected"]
#     ]

#     st.subheader(
#         "Detected Abnormalities"
#     )

#     if detected:

#         for result in detected:

#             st.success(
#                 f"**{result['Abnormality']}**  "
#                 f"— {result['Probability']:.1%}"
#             )

#     else:

#         st.info(
#             "No abnormalities detected "
#             "above the configured thresholds."
#         )


#     # -----------------------------------------
#     # All Predictions
#     # -----------------------------------------

#     st.subheader(
#         "All Predictions"
#     )

#     for result in results:

#         col1, col2, col3 = st.columns(
#             [3, 2, 2]
#         )

#         with col1:

#             st.write(
#                 f"**{result['Abnormality']}**"
#             )

#         with col2:

#             st.write(
#                 f"{result['Probability']:.2%}"
#             )

#         with col3:

#             if result["Detected"]:

#                 st.write("🟢 Detected")

#             else:

#                 st.write("⚪ Not Detected")

In [65]:
!mkdir -p /kaggle/working/models

In [66]:
!pip install -q streamlit pydicom opencv-python-headless
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 2s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
⠙

In [67]:
import os

print(os.path.exists("/kaggle/working/app.py"))

True


In [68]:
import subprocess
import urllib.request
import time

process = subprocess.Popen([
    "streamlit",
    "run",
    "/kaggle/working/app.py",
    "--server.port",
    "8501",
    "--server.headless",
    "true",
    "--server.enableCORS",
    "false",
    "--server.enableXsrfProtection",
    "false"
])

time.sleep(5)

print("Streamlit server started.")

2026-08-13 17:25:57.657 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.150.237.235:8501

Streamlit server started.


In [69]:
# external_ip = urllib.request.urlopen(
#     "https://ipv4.icanhazip.com"
# ).read().decode("utf8").strip()

# print("=========================================================")
# print(f"🔑 YOUR LOCALTUNNEL PASSWORD IS: {external_ip}")
# print("=========================================================")

In [70]:
# =========================================================
# STREAMLIT UI
# =========================================================

st.set_page_config(
    page_title="Knee MRI Abnormality Detection",
    page_icon="11",
    layout="wide"
)

# ---------------------------------------------------------
# Custom CSS
# ---------------------------------------------------------

st.markdown("""
<style>

.main-title {
    font-size: 42px;
    font-weight: 800;
    margin-bottom: 5px;
}

.subtitle {
    font-size: 17px;
    color: #6b7280;
    margin-bottom: 30px;
}

.section-title {
    font-size: 26px;
    font-weight: 700;
    margin-top: 30px;
    margin-bottom: 15px;
}

.detected-card {
    padding: 18px;
    border-radius: 12px;
    border: 1px solid #d1fae5;
    background-color: #ecfdf5;
    margin-bottom: 12px;
}

.not-detected-card {
    padding: 12px 16px;
    border-radius: 10px;
    border: 1px solid #e5e7eb;
    margin-bottom: 8px;
}

.probability {
    font-size: 24px;
    font-weight: 700;
}

.disease-name {
    font-size: 19px;
    font-weight: 700;
}

.small-text {
    color: #6b7280;
    font-size: 14px;
}

</style>
""", unsafe_allow_html=True)


# ---------------------------------------------------------
# Header
# ---------------------------------------------------------

st.markdown(
    '<div class="main-title">AA Knee MRI Abnormality Detection</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">'
    'Multi-Label Deep Learning Analysis of Knee MRI Studies'
    '</div>',
    unsafe_allow_html=True
)


# ---------------------------------------------------------
# Load Metadata
# ---------------------------------------------------------

series_meta_df = load_series_metadata(
    TEST_SERIES_META_PATH
)

series_lookup = build_series_lookup(
    series_meta_df
)


# ---------------------------------------------------------
# Get Available Studies
# ---------------------------------------------------------

study_folders = [
    folder
    for folder in TEST_SERIES_DIR.iterdir()
    if folder.is_dir()
]

study_uids = sorted([
    folder.name
    for folder in study_folders
])


# ---------------------------------------------------------
# Load Models
# ---------------------------------------------------------

models_list = load_models()


# ---------------------------------------------------------
# Study Input
# ---------------------------------------------------------

st.markdown(
    '<div class="section-title">🔎 Study Selection</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="small-text">'
    'Enter a StudyInstanceUID from the RSNA test dataset.'
    '</div>',
    unsafe_allow_html=True
)

selected_study_uid = st.text_input(
    "StudyInstanceUID",
    value=study_uids[0] if study_uids else "",
    label_visibility="collapsed"
)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

study_exists = (
    selected_study_uid in study_uids
)


if selected_study_uid:

    if study_exists:

        st.success(
            "✓ Study found in the test dataset."
        )

    else:

        st.error(
            "StudyInstanceUID not found in test_series."
        )


# ---------------------------------------------------------
# Analyze Button
# ---------------------------------------------------------

analyze = st.button(
    "🔍 Analyze Study",
    type="primary",
    use_container_width=True
)


# ---------------------------------------------------------
# Run Inference
# ---------------------------------------------------------

if analyze:

    if not selected_study_uid:

        st.error(
            "Please enter a StudyInstanceUID."
        )

        st.stop()


    if not study_exists:

        st.error(
            "The entered StudyInstanceUID does not exist "
            "in the RSNA test dataset."
        )

        st.stop()


    selected_study_path = (
        TEST_SERIES_DIR
        / selected_study_uid
    )


    # -----------------------------------------------------
    # Inference
    # -----------------------------------------------------

    with st.spinner(
        "Analyzing Knee MRI Study..."
    ):

        output = run_study_inference(
            selected_study_path,
            series_lookup,
            models_list
        )


    results = output["results"]

    detected = [
        result
        for result in results
        if result["Detected"]
    ]


    # -----------------------------------------------------
    # Study Information
    # -----------------------------------------------------

    st.markdown(
        '<div class="section-title">📋 Study Information</div>',
        unsafe_allow_html=True
    )

    col1, col2 = st.columns(2)

    with col1:

        st.metric(
            "StudyInstanceUID",
            selected_study_uid
        )

    with col2:

        st.metric(
            "Detected Abnormalities",
            len(detected)
        )


    # -----------------------------------------------------
    # Selected Series
    # -----------------------------------------------------

    st.markdown(
        '<div class="section-title">🧠 Selected MRI Series</div>',
        unsafe_allow_html=True
    )

    for i, series in enumerate(
        output["selected_series"],
        start=1
    ):

        st.write(
            f"**Series {i}:** "
            f"`{series['SeriesInstanceUID']}`"
        )


    # -----------------------------------------------------
    # Detected Abnormalities
    # -----------------------------------------------------

    st.markdown(
        '<div class="section-title">'
        '🚨 Detected Abnormalities'
        '</div>',
        unsafe_allow_html=True
    )


    if detected:

        for result in sorted(
            detected,
            key=lambda x: x["Probability"],
            reverse=True
        ):

            probability = (
                result["Probability"]
            )

            st.markdown(
                f"""
                <div class="detected-card">

                    <div class="disease-name">
                        🟢 {result["Abnormality"]}
                    </div>

                    <div class="probability">
                        {probability:.1%}
                    </div>

                    <div class="small-text">
                        Model confidence
                    </div>

                </div>
                """,
                unsafe_allow_html=True
            )

    else:

        st.success(
            "No abnormalities detected above "
            "the configured thresholds."
        )


    # -----------------------------------------------------
    # All Predictions
    # -----------------------------------------------------

    st.markdown(
        '<div class="section-title">'
        '📊 Complete Model Predictions'
        '</div>',
        unsafe_allow_html=True
    )


    for result in results:

        probability = (
            result["Probability"]
        )

        abnormality = (
            result["Abnormality"]
        )

        detected_status = (
            result["Detected"]
        )


        col1, col2, col3 = st.columns(
            [4, 2, 2]
        )


        with col1:

            st.write(
                f"**{abnormality}**"
            )


        with col2:

            st.write(
                f"{probability:.2%}"
            )


        with col3:

            if detected_status:

                st.write(
                    "🟢 Detected"
                )

            else:

                st.write(
                    "⚪ Not Detected"
                )


    # -----------------------------------------------------
    # Disclaimer
    # -----------------------------------------------------

    st.divider()

    st.caption(
        "⚠️ This application is a Deep Learning research/demo "
        "system. Predictions are model outputs and should not "
        "be considered a clinical diagnosis."
    )

2026-08-13 17:26:00.983 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:26:00.984 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:26:00.985 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:26:00.985 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:26:00.986 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:26:00.988 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:26:00.988 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 17:26:00.989 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [71]:
# تأكد أن الملف اتحدث
!grep -n "StudyInstanceUID" /kaggle/working/app.py | tail

907:    "Enter StudyInstanceUID",
931:            "StudyInstanceUID not found."
948:            "Please enter a valid StudyInstanceUID."


In [72]:
import os

os.system("pkill -f 'streamlit run' || true")
os.system("pkill -f 'localtunnel' || true")

  Stopping...


15

In [73]:
# import subprocess
# import time

# process = subprocess.Popen([
#     "streamlit",
#     "run",
#     "/kaggle/working/app.py",
#     "--server.port", "8501",
#     "--server.headless", "true",
#     "--server.enableCORS", "false",
#     "--server.enableXsrfProtection", "false"
# ])

# time.sleep(5)

In [74]:
# !npx localtunnel --port 8501

In [75]:
!curl ipv4.icanhazip.com

34.150.237.235


In [76]:
!find /kaggle/working -maxdepth 2 -type f | sort

/kaggle/working/app.py
/kaggle/working/.virtual_documents/__notebook_source__.ipynb


In [77]:
# with open("/kaggle/working/app.py", "r", encoding="utf-8") as f:
#     code = f.read()

# print(code)

In [78]:
%history -n -f /kaggle/working/notebook_history.py

In [79]:
with open("/kaggle/working/notebook_history.py", "r", encoding="utf-8") as f:
    print(f.read())

   1:
%%writefile /kaggle/working/app.py

import pathlib
import numpy as np
import pandas as pd
import cv2
import pydicom

import torch
import torch.nn as nn
import torchvision.models as models

import streamlit as st


# =========================================================
# CONFIG
# =========================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

COMP_ROOT = pathlib.Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

TEST_SERIES_DIR = COMP_ROOT / "test_series"

TEST_SERIES_META_PATH = COMP_ROOT / "test_series.csv"

MODEL_DIR = pathlib.Path(
    "/kaggle/input/models/amraldswqy135/knee-model2/pytorch/default/1"
)


TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

NUM_CLASSES = len(TARGETS)

THRESHOLDS = {
    name: 0.50
    for 

In [80]:
Stop

NameError: name 'Stop' is not defined

In [ ]:
########################################################

In [109]:
%%writefile /kaggle/working/app.py

import pathlib
import numpy as np
import pandas as pd
import cv2
import pydicom

import torch
import torch.nn as nn
import torchvision.models as models

import streamlit as st


# =========================================================
# CONFIG
# =========================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

COMP_ROOT = pathlib.Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

TEST_SERIES_DIR = COMP_ROOT / "test_series"

TEST_SERIES_META_PATH = COMP_ROOT / "test_series.csv"

MODEL_DIR = pathlib.Path(
    "/kaggle/input/models/amraldswqy135/knee-model2/pytorch/default/1"
)


TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

NUM_CLASSES = len(TARGETS)

THRESHOLDS = {
    name: 0.50
    for name in TARGETS
}

PLANE_PRIORITY = {
    "Sagittal": 3,
    "Coronal": 2,
    "Axial": 1
}


# =========================================================
# MODEL
# =========================================================

class AttentionPool(nn.Module):

    def __init__(self, feat_dim, hidden_dim=128):

        super().__init__()

        self.attn = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, mask=None):

        scores = self.attn(x).squeeze(-1)

        if mask is not None:

            scores = scores.masked_fill(
                mask == 0,
                float("-inf")
            )

        weights = torch.softmax(
            scores,
            dim=1
        )

        return torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )


def build_resnet50_backbone():

    resnet = models.resnet50(
        weights=None
    )

    return nn.Sequential(
        *list(resnet.children())[:-1]
    )


class KneeAbnormalityModelMIL(nn.Module):

    def __init__(self, num_classes=12):

        super().__init__()

        self.backbone = build_resnet50_backbone()

        self.slice_pool = AttentionPool(2048)

        self.series_pool = AttentionPool(2048)

        self.dropout = nn.Dropout(
            p=0.4
        )

        self.fc = nn.Linear(
            2048,
            num_classes
        )

    def forward(self, x, series_mask):

        b, s, k, c, h, w = x.shape

        x = x.view(
            b * s * k,
            c,
            h,
            w
        )

        features = self.backbone(x).view(
            b,
            s,
            k,
            2048
        )

        series_embeds = self.slice_pool(
            features.view(
                b * s,
                k,
                2048
            )
        ).view(
            b,
            s,
            2048
        )

        study_embed = self.series_pool(
            series_embeds,
            mask=series_mask
        )

        study_embed = self.dropout(
            study_embed
        )

        return self.fc(study_embed)


# =========================================================
# LOAD MODELS
# =========================================================

@st.cache_resource
def load_models():

    checkpoint_paths = [

        MODEL_DIR /
        "knee_model_mil_v3_fold0_best.pt",

        MODEL_DIR /
        "knee_model_mil_v3_fold1_best.pt"
    ]

    loaded_models = []

    for checkpoint_path in checkpoint_paths:

        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False
        )

        model = KneeAbnormalityModelMIL(
            num_classes=NUM_CLASSES
        ).to(DEVICE)

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        model.eval()

        loaded_models.append(model)

    return loaded_models


# =========================================================
# SERIES METADATA
# =========================================================

@st.cache_data
def load_series_metadata(path):

    return pd.read_csv(path)


def build_series_lookup(series_meta_df):

    lookup = {}

    for _, row in series_meta_df.iterrows():

        uid = str(
            row["SeriesInstanceUID"]
        )

        lookup[uid] = {

            "Fluid_Sensitive":
                row.get(
                    "Fluid_Sensitive",
                    0
                ) or 0,

            "Fat_Suppression":
                row.get(
                    "Fat_Suppression",
                    0
                ) or 0,

            "Anatomical_Plane":
                row.get(
                    "Anatomical_Plane",
                    ""
                )
        }

    return lookup


def score_series(
    series_uid,
    series_lookup
):

    meta = series_lookup.get(
        series_uid
    )

    if meta is None:

        return -1

    return (

        meta["Fluid_Sensitive"] * 10

        +

        meta["Fat_Suppression"] * 5

        +

        PLANE_PRIORITY.get(
            meta["Anatomical_Plane"],
            0
        )
    )


def select_best_series(
    series_folders,
    series_lookup
):

    best_by_plane = {}

    for series_path in series_folders:

        series_uid = series_path.name

        meta = series_lookup.get(
            series_uid
        )

        plane = (
            meta["Anatomical_Plane"]
            if meta
            else "Unknown"
        )

        score = score_series(
            series_uid,
            series_lookup
        )

        if (
            plane not in best_by_plane
            or
            score > best_by_plane[plane][0]
        ):

            best_by_plane[plane] = (
                score,
                series_path
            )

    selected = []

    for plane in [
        "Sagittal",
        "Coronal",
        "Axial"
    ]:

        if plane in best_by_plane:

            selected.append(
                best_by_plane[plane][1]
            )

    return selected[:3]


# =========================================================
# DICOM PROCESSING
# =========================================================

def get_sorted_dicom_files(
    series_path
):

    dicom_files = list(
        series_path.glob("*.dcm")
    )

    headers = []

    for file_path in dicom_files:

        try:

            ds = pydicom.dcmread(
                file_path,
                stop_before_pixels=True
            )

            if hasattr(
                ds,
                "InstanceNumber"
            ):

                key = int(
                    getattr(
                        ds,
                        "InstanceNumber",
                        0
                    )
                )

            else:

                key = float(
                    getattr(
                        ds,
                        "ImagePositionPatient",
                        [0, 0, 0]
                    )[2]
                )

            headers.append(
                (
                    key,
                    file_path
                )
            )

        except Exception:

            continue

    headers.sort(
        key=lambda x: x[0]
    )

    return [
        file_path
        for _, file_path in headers
    ]


def select_16_slices(
    dicom_files
):

    n = len(dicom_files)

    if n == 0:

        return []

    if n <= 16:

        return dicom_files

    indices = np.linspace(
        0,
        n - 1,
        num=16,
        dtype=int
    )

    return [
        dicom_files[i]
        for i in indices
    ]


def load_selected_images(
    selected_files
):

    images = []

    for file_path in selected_files:

        try:

            ds = pydicom.dcmread(
                file_path
            )

            image = (
                ds.pixel_array
                .astype(np.float32)
            )

            images.append(image)

        except Exception:

            images.append(None)

    valid = [
        image
        for image in images
        if image is not None
    ]

    if not valid:

        return None

    fill_shape = valid[0].shape

    images = [

        image
        if image is not None

        else np.zeros(
            fill_shape,
            dtype=np.float32
        )

        for image in images
    ]

    return np.array(images)


# =========================================================
# PREPROCESSING
# =========================================================

def preprocess_volume(volume):

    volume = volume.astype(
        np.float32
    )

    # 1st / 99th percentile
    p1, p99 = np.percentile(
        volume,
        (1, 99)
    )

    denom = (
        (p99 - p1)
        if (p99 - p1) > 1e-5
        else 1.0
    )

    # Clip
    volume = np.clip(
        volume,
        p1,
        p99
    )

    # Normalize 0 → 1
    volume = (
        volume - p1
    ) / denom

    # Resize
    volume = np.array(

        [
            cv2.resize(
                image,
                (224, 224),
                interpolation=cv2.INTER_AREA
            )

            for image in volume

        ],

        dtype=np.float32
    )

    return volume


def preprocess_study(
    study_path,
    series_lookup,
    max_series=3
):

    study_path = pathlib.Path(
        study_path
    )

    series_folders = [

        folder

        for folder in study_path.iterdir()

        if folder.is_dir()
    ]

    selected_series = select_best_series(
        series_folders,
        series_lookup
    )

    stack = np.zeros(
        (
            max_series,
            16,
            224,
            224
        ),
        dtype=np.float32
    )

    mask = np.zeros(
        (max_series,),
        dtype=np.float32
    )

    selected_info = []

    for i, series_path in enumerate(
        selected_series
    ):

        dicom_files = (
            get_sorted_dicom_files(
                series_path
            )
        )

        selected_files = (
            select_16_slices(
                dicom_files
            )
        )

        volume = load_selected_images(
            selected_files
        )

        if volume is None:

            continue

        volume = preprocess_volume(
            volume
        )

        # Ensure exactly 16 slices
        if volume.shape[0] < 16:

            padded = np.zeros(
                (
                    16,
                    224,
                    224
                ),
                dtype=np.float32
            )

            padded[
                :volume.shape[0]
            ] = volume

            volume = padded

        elif volume.shape[0] > 16:

            volume = volume[:16]

        stack[i] = volume

        mask[i] = 1.0

        selected_info.append({

            "SeriesInstanceUID":
                series_path.name
        })

    return (
        stack,
        mask,
        selected_info
    )


# =========================================================
# MODEL INPUT
# =========================================================

def prepare_model_input(
    stack,
    mask
):

    stack_tensor = torch.from_numpy(
        stack
    )

    # [3,16,224,224]
    # →
    # [3,16,3,224,224]

    stack_tensor = (

        stack_tensor
        .unsqueeze(2)
        .repeat(
            1,
            1,
            3,
            1,
            1
        )
    )

    # Add batch
    # →
    # [1,3,16,3,224,224]

    stack_tensor = (
        stack_tensor
        .unsqueeze(0)
    )

    mask_tensor = (
        torch.from_numpy(mask)
        .unsqueeze(0)
    )

    return (

        stack_tensor.to(DEVICE),

        mask_tensor.to(DEVICE)
    )


# =========================================================
# PREDICTION
# =========================================================

def predict_study(
    models_list,
    stack,
    mask
):

    volumes, series_mask = (
        prepare_model_input(
            stack,
            mask
        )
    )

    fold_probabilities = []

    with torch.no_grad():

        for model in models_list:

            logits = model(
                volumes,
                series_mask
            )

            probabilities = (
                torch.sigmoid(logits)
            )

            fold_probabilities.append(
                probabilities
                .cpu()
                .numpy()[0]
            )

    fold_probabilities = np.array(
        fold_probabilities
    )

    # Fold 0 + Fold 1
    ensemble_probabilities = np.mean(
        fold_probabilities,
        axis=0
    )

    return ensemble_probabilities


# =========================================================
# RULES / THRESHOLDS
# =========================================================

def apply_thresholds(
    probabilities
):

    results = []

    for class_name, probability in zip(
        TARGETS,
        probabilities
    ):

        threshold = THRESHOLDS[
            class_name
        ]

        detected = (
            probability >= threshold
        )

        results.append({

            "Abnormality":
                class_name,

            "Probability":
                float(probability),

            "Detected":
                bool(detected)
        })

    return results


# =========================================================
# COMPLETE STUDY INFERENCE
# =========================================================

def run_study_inference(
    study_path,
    series_lookup,
    models_list
):

    # 1. Preprocess
    stack, mask, selected_series = (
        preprocess_study(
            study_path,
            series_lookup,
            max_series=3
        )
    )

    # 2. Prediction
    probabilities = predict_study(
        models_list,
        stack,
        mask
    )

    # 3. Rules
    results = apply_thresholds(
        probabilities
    )

    return {

        "probabilities":
            probabilities,

        "results":
            results,

        "selected_series":
            selected_series,

        "series_mask":
            mask
    }


# =========================================================
# STREAMLIT UI
# =========================================================

st.set_page_config(
    page_title="Knee MRI Abnormality Detection",
    page_icon="🦴",
    layout="wide"
)

st.title(
    "🦴 Knee MRI Abnormality Detection"
)

st.caption(
    "Multi-Label Deep Learning Analysis "
    "of Knee MRI Studies"
)


# =========================================================
# LOAD DATA
# =========================================================

series_meta_df = load_series_metadata(
    TEST_SERIES_META_PATH
)

series_lookup = build_series_lookup(
    series_meta_df
)


study_folders = [

    folder

    for folder in TEST_SERIES_DIR.iterdir()

    if folder.is_dir()
]

study_uids = sorted([
    folder.name
    for folder in study_folders
])


# =========================================================
# LOAD MODELS
# =========================================================

models_list = load_models()


# =========================================================
# STUDY SELECTION
# =========================================================

st.subheader("🔎 Select MRI Study")

st.caption(
    "Choose a StudyInstanceUID from the RSNA test dataset."
)


# ---------------------------------------------------------
# Session State
# ---------------------------------------------------------

if "selected_study_uid" not in st.session_state:

    st.session_state.selected_study_uid = None


if "study_page" not in st.session_state:

    st.session_state.study_page = 0


# ---------------------------------------------------------
# Check Studies
# ---------------------------------------------------------

if not study_uids:

    st.error(
        "No studies were found in test_series."
    )

    st.stop()


# ---------------------------------------------------------
# Pagination
# ---------------------------------------------------------

STUDIES_PER_PAGE = 8

total_studies = len(study_uids)

total_pages = int(
    np.ceil(
        total_studies / STUDIES_PER_PAGE
    )
)


start_index = (
    st.session_state.study_page
    * STUDIES_PER_PAGE
)

end_index = min(
    start_index + STUDIES_PER_PAGE,
    total_studies
)


current_studies = study_uids[
    start_index:end_index
]


# ---------------------------------------------------------
# Study Cards
# ---------------------------------------------------------

st.markdown(
    f"**Available Studies:** {total_studies}"
)

st.markdown(
    f"Page "
    f"{st.session_state.study_page + 1} "
    f"of "
    f"{total_pages}"
)


columns = st.columns(2)


for i, study_uid in enumerate(
    current_studies
):

    column = columns[i % 2]

    with column:

        # Short display ID
        display_uid = study_uid

        if len(display_uid) > 38:

            display_uid = (
                display_uid[:18]
                + "..."
                + display_uid[-12:]
            )


        selected = (
            st.session_state.selected_study_uid
            == study_uid
        )


        if selected:

            st.success(
                f"🟢 Selected Study\n\n"
                f"`{display_uid}`"
            )

        else:

            st.info(
                f"🦴 Study {start_index + i + 1}\n\n"
                f"`{display_uid}`"
            )


        if st.button(
            "✓ Selected"
            if selected
            else "Select Study",
            key=f"select_study_{study_uid}",
            use_container_width=True
        ):

            st.session_state.selected_study_uid = (
                study_uid
            )

            st.rerun()


# ---------------------------------------------------------
# Pagination Controls
# ---------------------------------------------------------

st.divider()


prev_col, page_col, next_col = st.columns(
    [1, 2, 1]
)


with prev_col:

    if st.button(
        "← Previous",
        disabled=(
            st.session_state.study_page == 0
        ),
        use_container_width=True
    ):

        st.session_state.study_page -= 1

        st.rerun()


with page_col:

    st.markdown(
        f"<div style='text-align:center; "
        f"padding-top:8px;'>"
        f"<b>Page "
        f"{st.session_state.study_page + 1}"
        f" / {total_pages}</b>"
        f"</div>",
        unsafe_allow_html=True
    )


with next_col:

    if st.button(
        "Next →",
        disabled=(
            st.session_state.study_page
            >= total_pages - 1
        ),
        use_container_width=True
    ):

        st.session_state.study_page += 1

        st.rerun()


# ---------------------------------------------------------
# Selected Study
# ---------------------------------------------------------

if st.session_state.selected_study_uid:

    st.divider()

    st.markdown(
        "### ✅ Selected Study"
    )


    selected_study_uid = (
        st.session_state.selected_study_uid
    )


else:

    st.warning(
        "Please select a study before analysis."
    )

    selected_study_uid = None


# =========================================================
# ANALYZE
# =========================================================

# =========================================================
# ANALYZE STUDY
# =========================================================

if st.button(
    "🔍 Analyze Selected Study",
    type="primary",
    use_container_width=True
):

    if selected_study_uid is None:

        st.error(
            "Please select a Study first."
        )

        st.stop()


    selected_study_path = (
        TEST_SERIES_DIR
        / selected_study_uid
    )


    with st.spinner(
        "🧠 Analyzing Knee MRI Study..."
    ):

        output = run_study_inference(
            selected_study_path,
            series_lookup,
            models_list
        )


    # =====================================================
    # RESULTS
    # =====================================================

    results = output["results"]


    detected = [

        result

        for result in results

        if result["Detected"]

    ]


    st.divider()


    st.subheader(
        "🚨 Detected Abnormalities"
    )


    if detected:

        for result in sorted(
            detected,
            key=lambda x: x["Probability"],
            reverse=True
        ):

            st.success(
                f"🟢 **{result['Abnormality']}** "
                f"— {result['Probability']:.1%}"
            )

    else:

        st.info(
            "No abnormalities detected "
            "above the configured thresholds."
        )


    # =====================================================
    # SELECTED SERIES
    # =====================================================

    st.subheader(
        "🧠 Selected MRI Series"
    )


    for i, series in enumerate(
        output["selected_series"],
        start=1
    ):

        st.write(
            f"**Series {i}:** "
            f"`{series['SeriesInstanceUID']}`"
        )


    # =====================================================
    # ALL PREDICTIONS
    # =====================================================

    st.subheader(
        "📊 All Predictions"
    )


    for result in results:

        col1, col2, col3 = st.columns(
            [3, 2, 2]
        )


        with col1:

            st.write(
                f"**{result['Abnormality']}**"
            )


        with col2:

            st.write(
                f"{result['Probability']:.2%}"
            )


        with col3:

            if result["Detected"]:

                st.write(
                    "🟢 Detected"
                )

            else:

                st.write(
                    "⚪ Not Detected"
                )


 

Overwriting /kaggle/working/app.py


In [ ]:
# with open("/kaggle/working/app.py", "r", encoding="utf-8") as f:
#     lines = f.readlines()

# for i in range(1045, min(1075, len(lines))):
#     print(f"{i+1}: {lines[i]}", end="")

In [110]:
import os

print(
    os.path.exists(
        "/kaggle/working/app.py"
    )
)

True


In [111]:
import subprocess
import time

process = subprocess.Popen([
    "streamlit",
    "run",
    "/kaggle/working/app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])

time.sleep(5)

print("Streamlit started.")

2026-08-13 17:52:02.482 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.150.237.235:8501

Streamlit started.


In [ ]:
!npx localtunnel --port 8501